In [1]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - MACHINE LEARNING
# TASK 3.1 - LOAD CLEANED DATA
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load cleaned appointment data
df = pd.read_csv("/content/appointments_clean.csv")

print("CARELOOP - MACHINE LEARNING")
print("---------------------------")

print("Dataset shape:", df.shape)
print("Total appointments:", len(df))
print("Total no-shows:", df["no_show"].sum())
print("No-show percentage:",
      round(df["no_show"].mean() * 100, 1))

print("\nFirst five records:")
print(df.head())


CARELOOP - MACHINE LEARNING
---------------------------
Dataset shape: (990, 16)
Total appointments: 990
Total no-shows: 218
No-show percentage: 22.0

First five records:
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         1

In [2]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - MACHINE LEARNING
# TASK 3.1 - FEATURES, SPLIT AND BASELINE
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# STEP 1 - Select model features

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# STEP 2 - Convert reminder Yes/No into 1/0

X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# STEP 3 - Convert categories into numeric columns

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# STEP 4 - Define target variable
# 1 = No-Show, 0 = Attended

y = df["no_show"]

# STEP 5 - Train-test split
# Keep original row order before splitting

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# STEP 6 - Calculate baseline accuracy
# Baseline always predicts Attended (0)

baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = np.mean(
    baseline_predictions == y_test.to_numpy()
)

# STEP 7 - Display results

print("CARELOOP - TASK 3.1 RESULTS")
print("=" * 40)

print("\nTotal model features:", X.shape[1])
print("\nFeature names:")
print(X.columns.tolist())

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nNo-shows in test data:", int(y_test.sum()))

print("\nBaseline accuracy:",
      round(baseline_accuracy, 4))

# STEP 8 - Verify acceptance criteria

assert X.shape[1] == 11
assert len(X_train) == 792
assert len(X_test) == 198
assert int(y_test.sum()) == 49

print("\nTASK 3.1 COMPLETED SUCCESSFULLY!")


CARELOOP - TASK 3.1 RESULTS

Total model features: 11

Feature names:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training records: 792
Testing records: 198

No-shows in test data: 49

Baseline accuracy: 0.7525

TASK 3.1 COMPLETED SUCCESSFULLY!


In [3]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.2
# LOGISTIC REGRESSION
# ==========================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# STEP 1 - Standardize numeric features
# Fit the scaler ONLY on training data

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("STEP 1: Feature scaling completed")

# STEP 2 - Train logistic regression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_s, y_train)

print("STEP 2: Logistic regression trained")

# STEP 3 - Make predictions on test data

lr_predictions = lr_model.predict(X_test_s)

lr_probabilities = lr_model.predict_proba(
    X_test_s
)[:, 1]

# STEP 4 - Evaluate the model

lr_accuracy = accuracy_score(y_test, lr_predictions)

lr_precision = precision_score(
    y_test, lr_predictions, zero_division=0
)

lr_recall = recall_score(
    y_test, lr_predictions, zero_division=0
)

lr_f1 = f1_score(
    y_test, lr_predictions, zero_division=0
)

lr_auc = roc_auc_score(
    y_test, lr_probabilities
)

lr_cm = confusion_matrix(
    y_test, lr_predictions
)

print("\nLOGISTIC REGRESSION RESULTS")
print("=" * 45)

print("Accuracy:", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall:", round(lr_recall, 4))
print("F1 Score:", round(lr_f1, 4))
print("ROC-AUC:", round(lr_auc, 4))

print("\nConfusion Matrix:")
print(lr_cm)

# STEP 5 - Display model coefficients

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_[0]
})

print("\nMODEL COEFFICIENTS")
print(coefficients.to_string(index=False))

# STEP 6 - Analyze reminder_sent

reminder_index = X.columns.get_loc("reminder_sent")

reminder_coefficient = (
    lr_model.coef_[0][reminder_index]
)

# Odds multiplier for a one-standard-deviation increase
scaled_odds = np.exp(reminder_coefficient)

# Odds multiplier for actual No (0) to Yes (1)
original_coefficient = (
    reminder_coefficient /
    scaler.scale_[reminder_index]
)

actual_odds = np.exp(original_coefficient)

print("\nREMINDER SENT ANALYSIS")
print("=" * 45)

print(
    "Scaled coefficient:",
    round(reminder_coefficient, 4)
)

print(
    "Odds multiplier per standard deviation:",
    round(scaled_odds, 4)
)

print(
    "Odds multiplier for No to Yes:",
    round(actual_odds, 4)
)

print("\nTASK 3.2 COMPLETED SUCCESSFULLY!")


STEP 1: Feature scaling completed
STEP 2: Logistic regression trained

LOGISTIC REGRESSION RESULTS
Accuracy: 0.7879
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

Confusion Matrix:
[[139  10]
 [ 32  17]]

MODEL COEFFICIENTS
                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658

REMINDER SENT ANALYSIS
Scaled coefficient: -0.5807
Odds multiplier per standard deviation: 0.5595
Odds multiplier for No to Yes: 0.3094

TASK 3.2 COMPLETED SUCCESSFULLY!


In [4]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.3
# DECISION TREE AND OVERFITTING
# ==========================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# STEP 1 - Full Decision Tree (No depth limit)

full_tree = DecisionTreeClassifier(
    random_state=42
)

full_tree.fit(X_train, y_train)

# STEP 2 - Evaluate full tree

full_train_predictions = full_tree.predict(X_train)
full_test_predictions = full_tree.predict(X_test)

full_train_accuracy = accuracy_score(
    y_train, full_train_predictions
)

full_test_accuracy = accuracy_score(
    y_test, full_test_predictions
)

print("FULL DECISION TREE")
print("=" * 40)

print("Training accuracy:",
      round(full_train_accuracy, 4))

print("Testing accuracy:",
      round(full_test_accuracy, 4))

print("Tree depth:", full_tree.get_depth())

# STEP 3 - Pruned Decision Tree (Max Depth = 3)

depth3_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

depth3_tree.fit(X_train, y_train)

# STEP 4 - Evaluate pruned tree

depth3_train_predictions = depth3_tree.predict(X_train)
depth3_test_predictions = depth3_tree.predict(X_test)

depth3_train_accuracy = accuracy_score(
    y_train, depth3_train_predictions
)

depth3_test_accuracy = accuracy_score(
    y_test, depth3_test_predictions
)

print("\nPRUNED DECISION TREE (DEPTH = 3)")
print("=" * 40)

print("Training accuracy:",
      round(depth3_train_accuracy, 4))

print("Testing accuracy:",
      round(depth3_test_accuracy, 4))

print("Tree depth:", depth3_tree.get_depth())

# STEP 5 - Compare training and test accuracy gaps

full_gap = full_train_accuracy - full_test_accuracy

depth3_gap = depth3_train_accuracy - depth3_test_accuracy

print("\nOVERFITTING ANALYSIS")
print("=" * 40)

print("Full tree accuracy gap:",
      round(full_gap, 4))

print("Depth-3 tree accuracy gap:",
      round(depth3_gap, 4))

if full_gap > depth3_gap:
    print("\nThe full tree has a larger")
    print("training-test accuracy gap.")
    print("This indicates stronger overfitting.")
else:
    print("\nCheck both accuracy gaps")
    print("to assess generalization.")

print("\nTASK 3.3 COMPLETED SUCCESSFULLY!")


FULL DECISION TREE
Training accuracy: 1.0
Testing accuracy: 0.6717
Tree depth: 15

PRUNED DECISION TREE (DEPTH = 3)
Training accuracy: 0.822
Testing accuracy: 0.7778
Tree depth: 3

OVERFITTING ANALYSIS
Full tree accuracy gap: 0.3283
Depth-3 tree accuracy gap: 0.0442

The full tree has a larger
training-test accuracy gap.
This indicates stronger overfitting.

TASK 3.3 COMPLETED SUCCESSFULLY!


In [5]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.4
# RANDOM FOREST AND MODEL COMPARISON
# ==========================================

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

# STEP 1 - Build Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train using training data only
rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

# STEP 2 - Make test predictions

rf_predictions = rf_model.predict(X_test)

rf_probabilities = rf_model.predict_proba(
    X_test
)[:, 1]

# STEP 3 - Calculate performance metrics

rf_accuracy = accuracy_score(
    y_test, rf_predictions
)

rf_recall = recall_score(
    y_test, rf_predictions,
    zero_division=0
)

rf_auc = roc_auc_score(
    y_test, rf_probabilities
)

rf_train_accuracy = accuracy_score(
    y_train,
    rf_model.predict(X_train)
)

print("\nRANDOM FOREST RESULTS")
print("=" * 45)

print("Training accuracy:",
      round(rf_train_accuracy, 4))

print("Testing accuracy:",
      round(rf_accuracy, 4))

print("Recall:",
      round(rf_recall, 4))

print("ROC-AUC:",
      round(rf_auc, 4))

# STEP 4 - Feature importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTOP THREE IMPORTANT FEATURES")
print("=" * 45)

print(feature_importance.head(3).to_string(index=False))

# STEP 5 - Evaluate depth-3 tree

depth3_accuracy = accuracy_score(
    y_test,
    depth3_test_predictions
)

depth3_recall = recall_score(
    y_test,
    depth3_test_predictions,
    zero_division=0
)

depth3_probabilities = depth3_tree.predict_proba(
    X_test
)[:, 1]

depth3_auc = roc_auc_score(
    y_test,
    depth3_probabilities
)

# STEP 6 - Compare three models

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],

    "Accuracy": [
        lr_accuracy,
        depth3_accuracy,
        rf_accuracy
    ],

    "Recall": [
        lr_recall,
        depth3_recall,
        rf_recall
    ],

    "ROC-AUC": [
        lr_auc,
        depth3_auc,
        rf_auc
    ]
})

print("\nMODEL COMPARISON TABLE")
print("=" * 55)

print(
    comparison.round(3).to_string(index=False)
)

# STEP 7 - Reporting-only observation

print("\nTEST-SET COMPARISON (INFORMATION ONLY)")

reported_best = comparison.loc[
    comparison["ROC-AUC"].idxmax(), "Model"
]

print("Highest reported test ROC-AUC:", reported_best)

print(
    "Do not select the final model from test results."
)
print(
    "Formal selection will use training-only "
    "cross-validation in Task 3.5."
)

print("\nTASK 3.4 COMPLETED SUCCESSFULLY!")


Random Forest trained successfully!

RANDOM FOREST RESULTS
Training accuracy: 1.0
Testing accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057

TOP THREE IMPORTANT FEATURES
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739

MODEL COMPARISON TABLE
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression     0.788   0.347    0.789
Decision Tree (Depth 3)     0.778   0.224    0.700
          Random Forest     0.747   0.306    0.706

TEST-SET COMPARISON (INFORMATION ONLY)
Highest reported test ROC-AUC: Logistic Regression
Do not select the final model from test results.
Formal selection will use training-only cross-validation in Task 3.5.

TASK 3.4 COMPLETED SUCCESSFULLY!


In [6]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.5(a)
# RANDOM FOREST HYPERPARAMETER TUNING
# ==========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# STEP 1 - Define Random Forest

rf_tuning_model = RandomForestClassifier(
    random_state=42
)

# STEP 2 - Define parameter combinations

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# STEP 3 - Set up GridSearchCV
# Only training data will be used

grid_search = GridSearchCV(
    estimator=rf_tuning_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

# STEP 4 - Train and tune

print("Starting Random Forest tuning...")

grid_search.fit(X_train, y_train)

# STEP 5 - Display results

print("\nRANDOM FOREST TUNING RESULTS")
print("=" * 45)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation ROC-AUC:")
print(round(grid_search.best_score_, 4))

# STEP 6 - Store best model

best_rf_model = grid_search.best_estimator_

print("\nBest Random Forest model stored.")

print("\nTASK 3.5(a) COMPLETED SUCCESSFULLY!")


Starting Random Forest tuning...

RANDOM FOREST TUNING RESULTS
Best parameters:
{'max_depth': 3, 'n_estimators': 200}

Best cross-validation ROC-AUC:
0.7673

Best Random Forest model stored.

TASK 3.5(a) COMPLETED SUCCESSFULLY!


In [7]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 3.5(b) - FINAL MODEL SELECTION
# USING TRAINING CROSS-VALIDATION
# ==========================================

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# STEP 1 - Logistic Regression Pipeline
# Scaling happens independently inside each CV fold

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# STEP 2 - Define other models

dt_cv_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

rf_cv_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# STEP 3 - Cross-validation (training data only)

print("Starting 5-Fold Cross-Validation...")

lr_cv_auc = cross_val_score(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

dt_cv_auc = cross_val_score(
    dt_cv_model,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf_cv_model,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

# Use GridSearchCV result from Task 3.5(a)

tuned_rf_cv_auc = grid_search.best_score_

# STEP 4 - Create comparison table

cv_scores = {
    "Logistic Regression": lr_cv_auc,
    "Decision Tree (Depth 3)": dt_cv_auc,
    "Random Forest": rf_cv_auc,
    "Tuned Random Forest": tuned_rf_cv_auc
}

cv_comparison = pd.DataFrame({
    "Model": list(cv_scores.keys()),
    "CV ROC-AUC": list(cv_scores.values())
})

cv_comparison = cv_comparison.sort_values(
    "CV ROC-AUC",
    ascending=False
)

print("\nMODEL CROSS-VALIDATION COMPARISON")
print("=" * 50)

print(cv_comparison.round(4).to_string(index=False))

# STEP 5 - Select highest-scoring model

chosen_model_name = max(
    cv_scores,
    key=cv_scores.get
)

candidate_models = {
    "Logistic Regression": lr_pipe,
    "Decision Tree (Depth 3)": dt_cv_model,
    "Random Forest": rf_cv_model,
    "Tuned Random Forest": best_rf_model
}

chosen_model = candidate_models[chosen_model_name]

print("\nFINAL MODEL SELECTION")
print("=" * 50)

print("Selected model:", chosen_model_name)

print(
    "Selected CV ROC-AUC:",
    round(cv_scores[chosen_model_name], 4)
)

print("\nSelection based ONLY on training data.")
print("Test data was not used for model selection.")

print("\nTASK 3.5(b) COMPLETED SUCCESSFULLY!")


Starting 5-Fold Cross-Validation...

MODEL CROSS-VALIDATION COMPARISON
                  Model  CV ROC-AUC
    Logistic Regression      0.7737
    Tuned Random Forest      0.7673
          Random Forest      0.7466
Decision Tree (Depth 3)      0.6784

FINAL MODEL SELECTION
Selected model: Logistic Regression
Selected CV ROC-AUC: 0.7737

Selection based ONLY on training data.
Test data was not used for model selection.

TASK 3.5(b) COMPLETED SUCCESSFULLY!


In [8]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.5(c)
# RISK THRESHOLD OPTIMIZATION
# ==========================================

import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# STEP 1 - Generate out-of-fold predictions
# Training data only - do not use test data

print("Generating cross-validated risk scores...")

oof_probabilities = cross_val_predict(
    chosen_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Out-of-fold predictions generated!")

# STEP 2 - Compare two thresholds

thresholds = [0.50, 0.35]

threshold_results = []

for threshold in thresholds:

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_train,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        predictions,
        zero_division=0
    )

    flagged = int(predictions.sum())

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Patients Flagged": flagged
    })

# STEP 3 - Display comparison table

threshold_comparison = pd.DataFrame(
    threshold_results
)

print("\nRISK THRESHOLD COMPARISON")
print("=" * 65)

print(
    threshold_comparison.round(4).to_string(
        index=False
    )
)

# STEP 4 - Select higher F1 threshold

best_row = threshold_comparison.sort_values(
    ["F1 Score", "Recall"],
    ascending=[False, False]
).iloc[0]

chosen_threshold = float(
    best_row["Threshold"]
)

print("\nFINAL THRESHOLD SELECTION")
print("=" * 45)

print("Selected threshold:", chosen_threshold)

print(
    "Selected F1 Score:",
    round(float(best_row["F1 Score"]), 4)
)

print(
    "Selected Recall:",
    round(float(best_row["Recall"]), 4)
)

print(
    "Patients flagged in training:",
    int(best_row["Patients Flagged"])
)

print("\nWhy recall matters:")
print(
    "Missing a high-risk appointment may leave"
)
print(
    "an empty doctor slot, while an additional"
)
print(
    "follow-up call has a smaller operational cost."
)

print("\nSelection used training data only.")
print("Test data was not used for threshold selection.")

print("\nTASK 3.5(c) COMPLETED SUCCESSFULLY!")


Generating cross-validated risk scores...
Out-of-fold predictions generated!

RISK THRESHOLD COMPARISON
 Threshold  Precision  Recall  F1 Score  Patients Flagged
      0.50     0.5753  0.2485    0.3471                73
      0.35     0.5130  0.4675    0.4892               154

FINAL THRESHOLD SELECTION
Selected threshold: 0.35
Selected F1 Score: 0.4892
Selected Recall: 0.4675
Patients flagged in training: 154

Why recall matters:
Missing a high-risk appointment may leave
an empty doctor slot, while an additional
follow-up call has a smaller operational cost.

Selection used training data only.
Test data was not used for threshold selection.

TASK 3.5(c) COMPLETED SUCCESSFULLY!


In [9]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.5(d)
# FINAL MODEL TEST EVALUATION
# ==========================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# STEP 1 - Confirm selected model and threshold

print("FINAL MODEL EVALUATION")
print("=" * 45)

print("Selected model:", chosen_model_name)
print("Selected threshold:", chosen_threshold)

# STEP 2 - Fit selected model on full training data
# Logistic regression pipeline handles scaling

chosen_model.fit(X_train, y_train)

print("\nSelected model trained successfully!")

# STEP 3 - Predict probabilities on test data
# This is the formal final evaluation

test_probabilities = chosen_model.predict_proba(
    X_test
)[:, 1]

# STEP 4 - Apply selected threshold

test_predictions = (
    test_probabilities >= chosen_threshold
).astype(int)

# STEP 5 - Calculate evaluation metrics

final_accuracy = accuracy_score(
    y_test, test_predictions
)

final_precision = precision_score(
    y_test, test_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test, test_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test, test_predictions,
    zero_division=0
)

final_auc = roc_auc_score(
    y_test, test_probabilities
)

flagged_count = int(test_predictions.sum())

final_cm = confusion_matrix(
    y_test, test_predictions
)

# STEP 6 - Display final results

print("\nFINAL TEST RESULTS")
print("=" * 45)

print("Accuracy:", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall:", round(final_recall, 4))
print("F1 Score:", round(final_f1, 4))
print("ROC-AUC:", round(final_auc, 4))

print("\nTotal test appointments:", len(y_test))
print("Actual no-shows:", int(y_test.sum()))
print("Appointments flagged:", flagged_count)

print("\nConfusion Matrix:")
print(final_cm)

# STEP 7 - Validation

assert len(test_probabilities) == 198
assert len(test_predictions) == 198
assert int(y_test.sum()) == 49

print("\nTASK 3.5(d) COMPLETED SUCCESSFULLY!")


FINAL MODEL EVALUATION
Selected model: Logistic Regression
Selected threshold: 0.35

Selected model trained successfully!

FINAL TEST RESULTS
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889

Total test appointments: 198
Actual no-shows: 49
Appointments flagged: 49

Confusion Matrix:
[[128  21]
 [ 21  28]]

TASK 3.5(d) COMPLETED SUCCESSFULLY!


In [10]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 3 - TASK 3.5(e)
# EXPORT FINAL PATIENT RISK SCORES
# ==========================================

import pandas as pd
import os

# STEP 1 - Create risk score dataset
# Preserve alignment with X_test and y_test

risk_scores = pd.DataFrame({
    "appointment_id": df.loc[
        X_test.index, "appointment_id"
    ].to_numpy(),

    "no_show": y_test.to_numpy(),

    "risk_prob": test_probabilities,

    "flag": test_predictions
})

# STEP 2 - Validate risk scores

assert len(risk_scores) == 198

assert risk_scores["appointment_id"].is_unique

assert risk_scores["risk_prob"].between(
    0, 1
).all()

assert risk_scores["flag"].isin([0, 1]).all()

assert (
    risk_scores["flag"].to_numpy() ==
    (risk_scores["risk_prob"].to_numpy()
     >= chosen_threshold).astype(int)
).all()

# STEP 3 - Export CSV file

output_path = "/content/test_risk_scores.csv"

risk_scores.to_csv(
    output_path,
    index=False
)

# STEP 4 - Verify saved file

saved_data = pd.read_csv(output_path)

print("CARELOOP - FINAL RISK SCORE EXPORT")
print("=" * 45)

print("Selected model:", chosen_model_name)
print("Risk threshold:", chosen_threshold)

print("\nTotal exported rows:", len(saved_data))

print(
    "Appointments flagged:",
    int(saved_data["flag"].sum())
)

print(
    "Actual no-shows:",
    int(saved_data["no_show"].sum())
)

print(
    "File exists:",
    os.path.exists(output_path)
)

print("\nExported columns:")
print(saved_data.columns.tolist())

print("\nFirst five records:")
print(saved_data.head().to_string(index=False))

print("\nTASK 3.5(e) COMPLETED SUCCESSFULLY!")


CARELOOP - FINAL RISK SCORE EXPORT
Selected model: Logistic Regression
Risk threshold: 0.35

Total exported rows: 198
Appointments flagged: 49
Actual no-shows: 49
File exists: True

Exported columns:
['appointment_id', 'no_show', 'risk_prob', 'flag']

First five records:
appointment_id  no_show  risk_prob  flag
         A0217        0   0.021565     0
         A0336        0   0.150917     0
         A0508        0   0.518692     1
         A0314        0   0.191300     0
         A0089        0   0.167718     0

TASK 3.5(e) COMPLETED SUCCESSFULLY!
